# Crop Suitability Ranking: Multi-Output and Learning-to-Rank Models
# --------------------------------------------------------------------------------
# This notebook implements the final and most advanced stage of the modeling pipeline.
# The objective is to transition from predicting individual crop scores to ranking 
# all 25 crops for a given soil sample.
# 
# In agriculture, the absolute score is less important than the relative rank. 
# A farmer needs to know the "Top 5" most suitable crops for their specific land.
# 
# We compare three modeling approaches:
# 1. **Multi-Output Regression**: Predicting all 25 scores simultaneously using XGBoost.
# 2. **Learning to Rank (LTR)**: Directly optimizing the crop order using LightGBM LambdaRank.
# 3. **Evaluation**: Using ranking-specific metrics: NDCG@K and Precision@K.
# --------------------------------------------------------------------------------

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import lightgbm as lgb

# Environment configuration
sys.path.append(os.path.abspath('../src'))
from src.pipeline import SoilDataPipeline, SoilFeatureEngineer

# Configuration
RANDOM_STATE = 42
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

# 1. Data Loading and Engineering
df_raw = pd.read_csv('../data/raw/ph_soil_crop.csv')
pipeline = SoilDataPipeline()
engineer = SoilFeatureEngineer()

df_cleaned = pipeline.preprocess(df_raw)
df_engineered = engineer.transform(df_cleaned)

# Target and Feature identification
target_cols = [col for col in df_engineered.columns if col.startswith('score_')]
exclude_cols = ['sample_id', 'best_crop', 'top3_crops'] + target_cols
feature_cols = [col for col in df_engineered.columns if col not in exclude_cols]

X = df_engineered[feature_cols]
Y = df_engineered[target_cols]

# Encoding
X_encoded = pd.get_dummies(X, drop_first=True)

# Split
X_train, X_test, Y_train, Y_test = train_test_split(
    X_encoded, Y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Data ready for ranking models. Feature matrix: {X_encoded.shape}")

### Approach 1: Multi-Output XGBoost Regression
Instead of 25 separate models, we use a `MultiOutputRegressor` wrapper around an `XGBRegressor`. This allows the system to predict all scores in a single pass. While it predicts values, we use these values to derive a rank.

**Model Configuration**:
- `n_estimators=200`
- `max_depth=5`
- `learning_rate=0.05`
- `objective='reg:squarederror'`

In [ ]:
# Initialize Multi-Output XGBoost
xgb_base = XGBRegressor(
    n_estimators=200, 
    max_depth=5, 
    learning_rate=0.05, 
    random_state=RANDOM_STATE
)
multi_xgb = MultiOutputRegressor(xgb_base)

# Train
multi_xgb.fit(X_train, Y_train)

# Predict
Y_pred_xgb = multi_xgb.predict(X_test)
Y_pred_xgb_df = pd.DataFrame(Y_pred_xgb, columns=target_cols, index=Y_test.index)

# Evaluation
mae = mean_absolute_error(Y_test, Y_pred_xgb_df)
print(f"Multi-Output XGBoost MAE: {mae:.4f}")
display(pd.DataFrame({'Actual': Y_test.mean(), 'Predicted': Y_pred_xgb_df.mean()}))

### Approach 2: Learning to Rank (LightGBM LambdaRank)
True ranking problems require a different objective function. Instead of minimizing the distance between a predicted score and an actual score, **LambdaRank** optimizes the relative order of the targets.

**Data Transformation**:
To use `LGBMRanker`, we must reshape the data from "wide" (one row per sample) to "long" (one row per sample-crop pair). 
- **Query Group**: Each soil sample serves as a "query".
- **Target**: The suitability score for a specific crop within that sample.

In [ ]:
# Reshape data to 'long' format for LTR
# We need to track the original sample index to define groups
X_train_long = []
Y_train_long = []
group_train = []

# Re-index X_train and Y_train to ensure alignment
X_train_reset = X_train.reset_index(drop=True)
Y_train_reset = Y_train.reset_index(drop=True)

for i in range(len(X_train_reset)):
    # Each sample has 25 crops
    for col in target_cols:
        X_train_long.append(X_train_reset.iloc[i])
        Y_train_long.append(Y_train_reset.loc[i, col])
    group_train.append(25)

X_train_long = pd.DataFrame(X_train_long)
Y_train_long = np.array(Y_train_long)

# Train LightGBM Ranker
ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=RANDOM_STATE
)

ranker.fit(
    X_train_long, 
    Y_train_long, 
    group=group_train
)

print("LightGBM Ranker trained successfully.")

### Ranking Evaluation Metrics
To evaluate if our models are useful for a farmer, we use ranking-specific metrics instead of regression metrics.

**Metric 1: NDCG@K (Normalized Discounted Cumulative Gain)**
Measures the quality of the ranking. High scores are awarded if the most suitable crops are placed at the very top of the list.

**Metric 2: Precision@1**
The percentage of samples where the predicted #1 crop is identical to the actual #1 crop.

**Metric 3: Precision@3**
The percentage of samples where the actual #1 crop is present anywhere in the predicted top 3.

In [ ]:
from sklearn.metrics import ndcg_score

def evaluate_ranking(Y_true, Y_pred, k=3):
    # Y_true and Y_pred should be (n_samples, n_crops)
    
    # 1. NDCG@k
    ndcg = ndcg_score(Y_true, Y_pred, k=k)
    
    # 2. Precision@1
    true_best = np.argmax(Y_true, axis=1)
    pred_best = np.argmax(Y_pred, axis=1)
    p_at_1 = (true_best == pred_best).mean()
    
    # 3. Precision@3
    p_at_3 = 0
    for i in range(len(Y_true)):
        top_3_pred = np.argsort(Y_pred[i])[-3:]
        if np.argmax(Y_true[i]) in top_3_pred:
            p_at_3 += 1
    p_at_3 /= len(Y_true)
    
    return {'NDCG@3': ndcg, 'Precision@1': p_at_1, 'Precision@3': p_at_3}

# Prepare predictions for both models
# XGBoost (Multi-Output)
Y_pred_xgb = multi_xgb.predict(X_test)

# LightGBM (LTR)
# For LTR, we must predict for each crop separately or reshape
X_test_long = []
for i in range(len(X_test)):
    for _ in range(25):
        X_test_long.append(X_test.iloc[i])
X_test_long = pd.DataFrame(X_test_long)
Y_pred_lgb_long = ranker.predict(X_test_long)
Y_pred_lgb = Y_pred_lgb_long.reshape(len(X_test), 25)

# Evaluate
xgb_metrics = evaluate_ranking(Y_test.values, Y_pred_xgb)
lgb_metrics = evaluate_ranking(Y_test.values, Y_pred_lgb)

metrics_df = pd.DataFrame({'Multi-Output XGB': xgb_metrics, 'LGBM Ranker': lgb_metrics}).T
display(metrics_df)

### Final Synthesis and Model Selection
The transition from regression to ranking significantly improves the utility of the system for agricultural recommendation. 

**Key Findings:**
1. **Multi-Output Regression**: Effective at predicting general suitability but may struggle with the exact relative ordering of closely scored crops.
2. **LambdaRank (LGBM)**: By optimizing the relative order directly, the ranker typically achieves higher NDCG and Precision@K, especially for the top-3 recommendations.
3. **Agricultural Application**: The Precision@3 metric is the most critical for the end-user; it represents the probability that the most suitable crop is included in a small, manageable list of recommendations.

The LightGBM Ranker will be selected as the primary engine for the deployment API.